In [0]:
%sql
-- Ejecutar esto en AZ SQL > QUERY EDITOR
CREATE TABLE customers (
  ID INT PRIMARY KEY,
  Gender VARCHAR(10),
  Ever_Married VARCHAR(5),
  Age INT,
  Graduated VARCHAR(5),
  Profession VARCHAR(50),
  Work_Experience FLOAT NULL,
  Spending_Score VARCHAR(20),
  Family_Size FLOAT NULL,
  Var_1 VARCHAR(20),
  Segmentation VARCHAR(5)
);

In [0]:
jdbc_url = (
    "jdbc:sqlserver://server-az-sql-db-renzocavero.database.windows.net:1433;"
    "database=az-sql-db-renzocavero;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": dbutils.secrets.get(scope="kv-scope", key="sql-username"),
    "password": dbutils.secrets.get(scope="kv-scope", key="sql-pw"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("datalake", "adlsrenzocavero")
dbutils.widgets.text("csv", "customer_segmentation")

container = dbutils.widgets.get("container")
datalake = dbutils.widgets.get("datalake")
csv = dbutils.widgets.get("csv")

ruta = f"abfss://{container}@{datalake}.dfs.core.windows.net/{csv}.csv"
df = spark.read.option('header', True).option('inferSchema', True).csv(ruta)

In [0]:
df.write.jdbc(
    url=jdbc_url,
    table="dbo.customers",
    mode="overwrite",
    properties=connection_properties
)